# Input Routing Demo

This notebook creates demo files, routes PDF and image inputs, and shows OCR output.

In [ ]:
from pathlib import Path
import sys

import fitz
from PIL import Image, ImageDraw
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from input.input_handler import InputHandler
from ocr.ocr_engine import OCREngine

demo_dir = project_root / "notebooks" / "_demo_assets"
demo_dir.mkdir(exist_ok=True)
pdf_path = demo_dir / "sample_policy.pdf"
image_path = demo_dir / "sample_card.png"

document = fitz.open()
page_one = document.new_page()
page_one.insert_text(
    (72, 72),
    "\n".join(
        [
            "CLAIM SUMMARY",
            "Member Information",
            "Deductible: 500",
            "Copay: 25",
            "Coinsurance: 20%",
            "Plan Type: PPO",
        ]
    ),
)
page_two = document.new_page()
page_two.insert_text((72, 72), "BENEFITS\nEmergency Room: 250\nUrgent Care: 50")
document.save(pdf_path)
document.close()

image = Image.new("RGBA", (320, 180), (244, 248, 255, 255))
draw = ImageDraw.Draw(image)
draw.rectangle((20, 20, 300, 160), outline=(40, 70, 120), width=3)
draw.text((40, 70), "Sample image input", fill=(40, 70, 120))
image.save(image_path, format="PNG")

handler = InputHandler()
ocr = OCREngine(engine="paddle")
pdf_path, image_path

In [ ]:
# Cell 1: Load a PDF, show page count, and display page 1
loaded_pdf = handler.load(pdf_path)
print(f"PDF total pages: {loaded_pdf.total_pages}")
display(loaded_pdf.page_images[1].image)

In [ ]:
# Cell 2: Load an image and show it behaves like a one-page input
loaded_image = handler.load(image_path)
print(f"Image total pages: {loaded_image.total_pages}")
display(loaded_image.page_images[1].image)

In [ ]:
# Cell 3: Run OCR and inspect the structured output
structured_pages = ocr.process_pdf(pdf_path)
structured_pages[0]

In [ ]:
# Cell 4: Show index_text and confirm deductible values are preserved
print(structured_pages[0].index_text)
assert "500" in structured_pages[0].index_text
assert "25" in structured_pages[0].index_text
assert "20%" in structured_pages[0].index_text